# 04 — Performance Monitor & Scraping Statistics

Owner: Member 4.
Measures performance (RAM, disk storage, execution wall time) during the crawling pipeline and computes key scraping statistics from `logs/papers.csv` and `logs/references.csv`.

Provides data structures and plotting helpers for the Jupyter Book milestone report.

In [ ]:
if "CONFIG_LOADED" not in globals():
    get_ipython().run_line_magic("run", "-i ./00_config.ipynb")

import csv
import json
import os
import shutil
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import psutil


## 1. Directory size and storage helper

Calculates the actual storage footprint of directories (e.g. `DATA_DIR`) in an auditable way.

In [ ]:
def get_dir_size_bytes(dir_path):
    """Recursively calculate the total size of all files in a directory in bytes."""
    path = Path(dir_path)
    if not path.exists():
        return 0
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total


def get_dir_size_mb(dir_path):
    """Directory size in megabytes (MB)."""
    return round(get_dir_size_bytes(dir_path) / (1024 * 1024), 2)


## 2. Auditable Performance Monitor

`PerformanceMonitor` runs a daemon background thread that samples process RAM usage periodically (`psutil.Process().memory_info().rss`), tracks peak/average RAM, monitors storage growth in `DATA_DIR`, and records stage durations (such as entry discovery time and total wall-clock time).

In [ ]:
PERF_LOG = LOG_DIR / "performance.json"


class PerformanceMonitor:
    """
    Background performance monitor tracking RAM, disk footprint, and execution wall time.
    """

    def __init__(self, sample_interval=0.5, data_dir=None, log_dir=None):
        self.sample_interval = sample_interval
        self.data_dir = Path(data_dir) if data_dir else DATA_DIR
        self.log_dir = Path(log_dir) if log_dir else LOG_DIR
        self._process = psutil.Process(os.getpid())

        self._stop_event = threading.Event()
        self._thread = None

        self.start_wall_time = None
        self.end_wall_time = None

        # Time-series RAM samples: list of (relative_seconds, ram_mb)
        self.ram_samples = []
        self.peak_ram_mb = 0.0

        # Storage footprint
        self.initial_disk_bytes = 0
        self.final_disk_bytes = 0
        self.max_disk_bytes = 0

        # Stage timings (e.g., discovery, downloads, references)
        self.stages = {}
        self._stage_starts = {}

    def _sample_loop(self):
        disk_check_counter = 0
        while not self._stop_event.is_set():
            try:
                # RSS memory in MB
                mem_mb = self._process.memory_info().rss / (1024 * 1024)
                rel_sec = time.time() - self.start_wall_time
                self.ram_samples.append((rel_sec, mem_mb))
                if mem_mb > self.peak_ram_mb:
                    self.peak_ram_mb = mem_mb

                # Sample disk usage periodically (every ~5 intervals) to avoid excessive I/O
                disk_check_counter += 1
                if disk_check_counter % 5 == 0:
                    current_disk = get_dir_size_bytes(self.data_dir)
                    if current_disk > self.max_disk_bytes:
                        self.max_disk_bytes = current_disk
            except Exception:
                pass
            self._stop_event.wait(self.sample_interval)

    def start(self):
        """Start tracking execution wall time and launch the RAM sampling background thread."""
        self.start_wall_time = time.time()
        self.initial_disk_bytes = get_dir_size_bytes(self.data_dir)
        self.max_disk_bytes = self.initial_disk_bytes

        self._stop_event.clear()
        self._thread = threading.Thread(target=self._sample_loop, daemon=True)
        self._thread.start()

    def start_stage(self, stage_name):
        """Mark the beginning of a specific sub-stage (e.g., 'discovery')."""
        self._stage_starts[stage_name] = time.time()

    def end_stage(self, stage_name):
        """Mark the completion of a specific sub-stage and record its duration."""
        if stage_name in self._stage_starts:
            duration = time.time() - self._stage_starts[stage_name]
            self.stages[stage_name] = round(duration, 3)
            return duration
        return 0.0

    def stop(self):
        """Stop tracking and finalize calculations."""
        self.end_wall_time = time.time()
        self._stop_event.set()
        if self._thread and self._thread.is_alive():
            self._thread.join(timeout=2.0)

        self.final_disk_bytes = get_dir_size_bytes(self.data_dir)
        if self.final_disk_bytes > self.max_disk_bytes:
            self.max_disk_bytes = self.final_disk_bytes

    def summary(self):
        """Return a structured dictionary of performance metrics."""
        total_time = (self.end_wall_time or time.time()) - (self.start_wall_time or time.time())
        mems = [s[1] for s in self.ram_samples]
        avg_ram = sum(mems) / len(mems) if mems else 0.0

        return {
            "start_time": to_iso(datetime.fromtimestamp(self.start_wall_time, tz=timezone.utc)) if self.start_wall_time else "",
            "end_time": to_iso(datetime.fromtimestamp(self.end_wall_time, tz=timezone.utc)) if self.end_wall_time else "",
            "total_wall_time_seconds": round(total_time, 2),
            "peak_ram_mb": round(self.peak_ram_mb, 2),
            "avg_ram_mb": round(avg_ram, 2),
            "initial_disk_mb": round(self.initial_disk_bytes / (1024 * 1024), 2),
            "max_disk_mb": round(self.max_disk_bytes / (1024 * 1024), 2),
            "final_disk_mb": round(self.final_disk_bytes / (1024 * 1024), 2),
            "samples_count": len(self.ram_samples),
            "stages": self.stages,
        }

    def save_summary(self, path=None):
        """Save the performance metrics as JSON for the Jupyter Book report."""
        out_path = Path(path) if path else PERF_LOG
        summary_data = self.summary()
        write_json(out_path, summary_data)
        return summary_data


## 3. Scraping Statistics Calculator

Reads `logs/papers.csv` (and `logs/references.csv`) to compute the exact statistics required by the Milestone 1 statement:
1. Number of papers scraped successfully & overall success rate
2. Average paper size (bytes) before and after figure removal, plus space savings (%)
3. Average number of references per paper (total & arXiv references)
4. Reference metadata scraping success rate
5. Running time: total time, average time per paper, entry discovery duration
6. Memory footprint: peak RAM, average RAM, max disk storage, final output storage

In [ ]:
def load_papers_dataframe(log_path=None):
    """Load logs/papers.csv into a typed pandas DataFrame."""
    path = Path(log_path) if log_path else PAPER_LOG
    if not path.exists():
        return pd.DataFrame(columns=LOG_FIELDS)
    df = pd.read_csv(path)
    numeric_cols = ["versions", "versions_ok", "size_before", "size_after", "n_refs", "n_arxiv_refs", "seconds"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df


def load_references_dataframe(log_path=None):
    """Load logs/references.csv into a pandas DataFrame."""
    path = Path(log_path) if log_path else LOG_DIR / "references.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def compute_scraping_statistics(papers_df=None, perf_summary=None, total_assigned_ids=None):
    """
    Compute comprehensive milestone statistics from the papers DataFrame and performance summary.
    """
    if papers_df is None:
        papers_df = load_papers_dataframe()

    total_logged = len(papers_df)
    total_assigned = total_assigned_ids or total_logged

    # Status counts
    ok_papers = papers_df[papers_df["status"] == "ok"]
    n_ok = len(ok_papers)
    success_rate = (n_ok / total_assigned * 100) if total_assigned > 0 else 0.0

    # Paper sizes (for papers with downloaded source files)
    downloaded = papers_df[papers_df["size_before"] > 0]
    avg_size_before = downloaded["size_before"].mean() if len(downloaded) > 0 else 0.0
    avg_size_after = downloaded["size_after"].mean() if len(downloaded) > 0 else 0.0
    space_saved_pct = ((1.0 - (avg_size_after / avg_size_before)) * 100) if avg_size_before > 0 else 0.0

    # References
    ref_checked = papers_df[papers_df["status"].isin(["ok", "partial"])]
    avg_refs = ref_checked["n_refs"].mean() if len(ref_checked) > 0 else 0.0
    avg_arxiv_refs = ref_checked["n_arxiv_refs"].mean() if len(ref_checked) > 0 else 0.0

    # Reference metadata success rate
    ref_df = load_references_dataframe()
    if len(ref_df) > 0 and "status" in ref_df.columns:
        ref_ok = len(ref_df[ref_df["status"] == "ok"])
        ref_success_rate = (ref_ok / len(ref_df) * 100)
    else:
        ref_success_rate = 100.0 if n_ok > 0 else 0.0

    # Timing
    avg_time_per_paper = papers_df["seconds"].mean() if total_logged > 0 else 0.0
    total_paper_time = papers_df["seconds"].sum() if total_logged > 0 else 0.0

    # Versions distribution
    version_counts = papers_df["versions"].value_counts().sort_index().to_dict() if "versions" in papers_df.columns else {}

    # Performance summary integration
    perf = perf_summary or {}
    discovery_time = perf.get("stages", {}).get("discovery", 0.0)
    total_wall_time = perf.get("total_wall_time_seconds", total_paper_time)
    peak_ram = perf.get("peak_ram_mb", 0.0)
    avg_ram = perf.get("avg_ram_mb", 0.0)
    max_disk = perf.get("max_disk_mb", 0.0)
    final_disk = perf.get("final_disk_mb", get_dir_size_mb(DATA_DIR))

    stats = {
        "papers_assigned": total_assigned,
        "papers_logged": total_logged,
        "papers_scraped_ok": n_ok,
        "overall_success_rate_pct": round(success_rate, 2),
        "avg_size_before_bytes": round(avg_size_before, 2),
        "avg_size_after_bytes": round(avg_size_after, 2),
        "space_saved_pct": round(space_saved_pct, 2),
        "avg_refs_per_paper": round(avg_refs, 2),
        "avg_arxiv_refs_per_paper": round(avg_arxiv_refs, 2),
        "ref_metadata_success_rate_pct": round(ref_success_rate, 2),
        "avg_time_per_paper_seconds": round(avg_time_per_paper, 2),
        "discovery_time_seconds": round(discovery_time, 2),
        "total_wall_time_seconds": round(total_wall_time, 2),
        "peak_ram_mb": peak_ram,
        "avg_ram_mb": avg_ram,
        "max_disk_mb": max_disk,
        "final_disk_mb": final_disk,
        "versions_distribution": version_counts,
    }
    return stats


def format_statistics_table(stats):
    """Format key scraping statistics into a clean DataFrame for report display."""
    rows = [
        ("Papers Targeted / Assigned", stats["papers_assigned"]),
        ("Papers Scraped Successfully", stats["papers_scraped_ok"]),
        ("Overall Success Rate", f"{stats['overall_success_rate_pct']}%"),
        ("Avg Size Before Removing Figures", f"{stats['avg_size_before_bytes']:,.0f} B ({stats['avg_size_before_bytes'] / 1024:.1f} KB)"),
        ("Avg Size After Removing Figures", f"{stats['avg_size_after_bytes']:,.0f} B ({stats['avg_size_after_bytes'] / 1024:.1f} KB)"),
        ("Storage Saved by Removing Figures", f"{stats['space_saved_pct']}%"),
        ("Avg References per Paper", f"{stats['avg_refs_per_paper']:.2f}"),
        ("Avg arXiv References per Paper", f"{stats['avg_arxiv_refs_per_paper']:.2f}"),
        ("Reference Extraction Success Rate", f"{stats['ref_metadata_success_rate_pct']}%"),
        ("Avg Processing Time per Paper", f"{stats['avg_time_per_paper_seconds']:.2f} s"),
        ("Entry Discovery Duration", f"{stats['discovery_time_seconds']:.2f} s"),
        ("Total Wall Time", f"{stats['total_wall_time_seconds']:.2f} s"),
        ("Peak RAM Consumption", f"{stats['peak_ram_mb']:.2f} MB"),
        ("Average RAM Consumption", f"{stats['avg_ram_mb']:.2f} MB"),
        ("Maximum Disk Storage Required", f"{stats['max_disk_mb']:.2f} MB"),
        ("Final Output Storage Size", f"{stats['final_disk_mb']:.2f} MB"),
    ]
    return pd.DataFrame(rows, columns=["Metric", "Value"])


## 4. Visualizations for the Jupyter Book Report

Plotting functions that generate publishable, dynamic figures during Jupyter Book compilation (as required: generated during build, not pasted screenshots).

In [ ]:
def plot_ram_usage(samples, peak_ram=None, avg_ram=None, save_path=None):
    """Plot RAM usage (MB) over time with peak and average annotations."""
    if not samples:
        return
    times = [s[0] for s in samples]
    mems = [s[1] for s in samples]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(times, mems, color="#2b5c8f", lw=2, label="RAM Consumption (RSS)")

    p_ram = peak_ram or max(mems)
    a_ram = avg_ram or (sum(mems) / len(mems))

    ax.axhline(p_ram, color="#d9534f", linestyle="--", label=f"Peak RAM: {p_ram:.1f} MB")
    ax.axhline(a_ram, color="#f0ad4e", linestyle=":", label=f"Avg RAM: {a_ram:.1f} MB")

    ax.set_title("RAM Consumption Over Crawling Pipeline", fontsize=13, fontweight="bold")
    ax.set_xlabel("Elapsed Time (seconds)", fontsize=11)
    ax.set_ylabel("Memory (MB)", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper right")
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300)
    plt.show()


def plot_size_comparison(papers_df, save_path=None):
    """Bar chart comparing average paper size before and after figure removal."""
    downloaded = papers_df[papers_df["size_before"] > 0]
    if len(downloaded) == 0:
        return

    avg_before_kb = downloaded["size_before"].mean() / 1024
    avg_after_kb = downloaded["size_after"].mean() / 1024

    fig, ax = plt.subplots(figsize=(6, 4.5))
    bars = ax.bar(["Before Figure Removal", "After Figure Removal"], [avg_before_kb, avg_after_kb],
                  color=["#e06666", "#6aa84f"], width=0.5)

    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.1f} KB",
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontweight="bold")

    saved_pct = ((1.0 - (avg_after_kb / avg_before_kb)) * 100) if avg_before_kb > 0 else 0
    ax.set_title(f"Average Paper Size Reduction (Saved {saved_pct:.1f}%)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Size (KB)", fontsize=11)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300)
    plt.show()


def plot_versions_distribution(papers_df, save_path=None):
    """Bar plot of papers by number of versions."""
    if "versions" not in papers_df.columns or len(papers_df) == 0:
        return

    counts = papers_df["versions"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar([f"v{v}" for v in counts.index], counts.values, color="#45818e", width=0.6)

    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h}",
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom")

    ax.set_title("Distribution of Paper Versions", fontsize=12, fontweight="bold")
    ax.set_xlabel("Revisions on arXiv", fontsize=11)
    ax.set_ylabel("Number of Papers", fontsize=11)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300)
    plt.show()


def plot_references_distribution(papers_df, save_path=None):
    """Histogram of references count per paper."""
    valid = papers_df[papers_df["n_refs"] >= 0]
    if len(valid) == 0:
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(valid["n_refs"], bins=15, alpha=0.7, color="#3d85c6", label="All References (S2)")
    ax.hist(valid["n_arxiv_refs"], bins=15, alpha=0.7, color="#674ea7", label="arXiv References Only")

    ax.set_title("Distribution of References per Paper", fontsize=12, fontweight="bold")
    ax.set_xlabel("Number of References", fontsize=11)
    ax.set_ylabel("Frequency", fontsize=11)
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300)
    plt.show()


## 5. Offline Test

Runs only when this notebook is opened on its own (`RUN_TEST = True`). Verifies the monitor's background sampling thread, summary metrics generation, and statistical calculations with simulated test records.

In [ ]:
if RUN_TEST:
    print("Testing PerformanceMonitor...")
    mon = PerformanceMonitor(sample_interval=0.1)
    mon.start()
    mon.start_stage("discovery")
    time.sleep(0.2)
    mon.end_stage("discovery")

    # Simulate some memory allocation
    dummy_block = [bytearray(1024 * 1024) for _ in range(10)]
    time.sleep(0.4)
    del dummy_block

    mon.stop()
    perf = mon.summary()
    print("Monitor summary:", perf)

    assert perf["total_wall_time_seconds"] >= 0.5
    assert perf["peak_ram_mb"] > 0
    assert perf["samples_count"] >= 3
    assert "discovery" in perf["stages"]

    print("Testing compute_scraping_statistics on mock DataFrame...")
    mock_df = pd.DataFrame([
        {"id": "2609.00001", "versions": 2, "versions_ok": 2, "size_before": 1000000, "size_after": 200000, "n_refs": 30, "n_arxiv_refs": 15, "status": "ok", "seconds": 6.2},
        {"id": "2609.00002", "versions": 1, "versions_ok": 1, "size_before": 500000, "size_after": 100000, "n_refs": 20, "n_arxiv_refs": 10, "status": "ok", "seconds": 3.8},
        {"id": "2609.00003", "versions": 1, "versions_ok": 0, "size_before": 0, "size_after": 0, "n_refs": 0, "n_arxiv_refs": 0, "status": "not_found", "seconds": 0.5},
    ])
    stats = compute_scraping_statistics(mock_df, perf_summary=perf, total_assigned_ids=3)
    table = format_statistics_table(stats)
    print("\nSummary Statistics Table:")
    print(table.to_string(index=False))

    assert stats["papers_assigned"] == 3
    assert stats["papers_scraped_ok"] == 2
    assert stats["overall_success_rate_pct"] == round(2 / 3 * 100, 2)
    assert stats["space_saved_pct"] == 80.0
    assert stats["avg_refs_per_paper"] == 25.0
    assert stats["avg_arxiv_refs_per_paper"] == 12.5

    print("\nAll Member 4 tests passed successfully!")
